In [ ]:
# ==========================================
# CELL 1: Imports and Data Loading
# ==========================================
import numpy as np
import torch

# Load the DeepMind data archive (Make sure factorizations_r.npz is in the same folder!)
print("Loading AlphaTensor data...")
data = np.load('factorizations_r.npz', allow_pickle=True)

# We are targeting the 3x3 matrix multiplication (the '3,3,3' tensor)
target_key = '3,3,3'

# Extract the U, V, and W matrices for the 23-step algorithm
U_23 = data[target_key].item()['u']
V_23 = data[target_key].item()['v']
W_23 = data[target_key].item()['w']

print(f"Successfully loaded! U matrix shape: {U_23.shape}")
# Expected output: (9, 23)


# ==========================================
# CELL 2: Target Tensor Setup
# ==========================================
# Build the "Perfect" 3D Tensor for 3x3 Matrix Multiplication
# Shape is (9, 9, 9) representing the flattened A, B, and C matrices.
T_target = torch.zeros((9, 9, 9), dtype=torch.float32)

for i in range(3):
    for j in range(3):
        for k in range(3):
            a_idx = i * 3 + j  # Flattened index for Matrix A
            b_idx = j * 3 + k  # Flattened index for Matrix B
            c_idx = i * 3 + k  # Flattened index for Matrix C
            T_target[a_idx, b_idx, c_idx] = 1.0

# Detect your GTX 1080 Ti and move the target tensor to the GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
T_target = T_target.to(device)


# ==========================================
# CELL 3: Compression and Optimizer Setup
# ==========================================
# Convert numpy arrays to PyTorch tensors
U_tensor = torch.tensor(U_23, dtype=torch.float32)
V_tensor = torch.tensor(V_23, dtype=torch.float32)
W_tensor = torch.tensor(W_23, dtype=torch.float32)

# Calculate the "strength" of each of the 23 columns
column_strengths = torch.norm(U_tensor, dim=0) * torch.norm(V_tensor, dim=0) * torch.norm(W_tensor, dim=0)

# Find the index of the weakest column and DROP IT to force 22 steps
weakest_idx = torch.argmin(column_strengths)
print(f"Dropping column index: {weakest_idx} to compress to 22 steps.")
indices_to_keep = [i for i in range(23) if i != weakest_idx]

# Create the new 22-step starting matrices and send to GPU
# requires_grad=True tells PyTorch that the Optimizer is allowed to modify these numbers
U_22 = U_tensor[:, indices_to_keep].clone().to(device).requires_grad_(True)
V_22 = V_tensor[:, indices_to_keep].clone().to(device).requires_grad_(True)
W_22 = W_tensor[:, indices_to_keep].clone().to(device).requires_grad_(True)

# Set up the Adam Optimizer (Adjust learning rate 'lr' if it gets stuck)
optimizer = torch.optim.Adam([U_22, V_22, W_22], lr=0.01)
print("Optimizer ready!")


# ==========================================
# CELL 4: The Training Loop
# ==========================================
epochs = 50000
print("Starting optimization loop...")

for epoch in range(epochs):
    optimizer.zero_grad()

    # 1. Reconstruct the 3D tensor using our 22 remaining steps
    # Einstein Summation (einsum) multiplies these matrices blazingly fast on your GPU
    T_pred = torch.einsum('ir,jr,kr->ijk', U_22, V_22, W_22)

    # 2. Calculate the Loss (Mean Squared Error)
    loss = torch.nn.functional.mse_loss(T_pred, T_target)

    # 3. Add an L1 Penalty
    # This acts as a mathematical pressure forcing decimals to become clean 0s, 1s, or -1s
    l1_penalty = 0.01 * (torch.abs(U_22).mean() + torch.abs(V_22).mean() + torch.abs(W_22).mean())
    total_loss = loss + l1_penalty

    # 4. Backpropagation (The GPU calculates how to improve the matrices)
    total_loss.backward()
    optimizer.step()

    # Print progress every 1000 steps
    if epoch % 1000 == 0:
        print(f"Epoch {epoch} | MSE Loss: {loss.item():.6f}")

    # THE WIN CONDITION (Loss effectively hits zero)
    if loss.item() < 1e-6:
        print(f"\n🎉 SUCCESS! You found a 22-step algorithm at Epoch {epoch}! 🎉")
        break

print("\nTraining run finished.")